In [1]:
import requests

# 1. Define the ID for Human Hemoglobin
pdb_id = "4HHB"

# 2. Point to the official structural database download URL
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"

print(f"Connecting to RCSB to fetch {pdb_id}...")
response = requests.get(url)

# 3. If the server says OK (200), save the text contents as a .pdb file
if response.status_code == 200:
    with open(f"{pdb_id}.pdb", "w", encoding="utf-8") as file:
        file.write(response.text)
    print(f"Success! '{pdb_id}.pdb' has been downloaded to your directory.")
else:
    print(f"Error: Could not retrieve file (Status code: {response.status_code})")

Connecting to RCSB to fetch 4HHB...
Success! '4HHB.pdb' has been downloaded to your directory.


In [2]:
import pandas as pd

# Define the file path (assumes 4HHB.pdb is in the same folder)
pdb_filename = "4HHB.pdb"

# List to hold our extracted atom data
atom_data = []

print(f"Reading {pdb_filename}...")

# 1. Use basic file handling to open and filter the file line-by-line
with open(pdb_filename, "r", encoding="utf-8") as file:
    for line in file:
        # PDB coordinate lines strictly start with the keyword 'ATOM  '
        if line.startswith("ATOM  "):
            
            # 2. Slice the string by specific PDB character columns
            # Column indices are strictly standard across all PDB files:
            atom_type = line[0:6].strip()
            atom_serial = int(line[6:11].strip())
            atom_name = line[12:16].strip()
            res_name = line[17:20].strip()
            chain_id = line[21].strip()
            res_seq = int(line[22:26].strip())
            
            # The 3D coordinates (X, Y, Z)
            x = float(line[30:38].strip())
            y = float(line[38:46].strip())
            z = float(line[46:54].strip())
            
            element = line[76:78].strip()

            # Append the structured data as a dictionary
            atom_data.append({
                "Type": atom_type,
                "Atom_ID": atom_serial,
                "Atom_Name": atom_name,
                "Residue": res_name,
                "Chain": chain_id,
                "Residue_Num": res_seq,
                "X": x,
                "Y": y,
                "Z": z,
                "Element": element
            })

# 3. Convert our clean list into a Pandas DataFrame
df = pd.DataFrame(atom_data)

print(f"Extraction complete! Found {len(df)} ATOM coordinate rows.\n")

# Display the first 10 rows of our structured coordinate data
print(df.head(10).to_string(index=False))

# Optional: Save the clean coordinates to a CSV file for future use
# df.to_csv("4HHB_clean_coordinates.csv", index=False)

Reading 4HHB.pdb...
Extraction complete! Found 4384 ATOM coordinate rows.

Type  Atom_ID Atom_Name Residue Chain  Residue_Num      X      Y      Z Element
ATOM        1         N     VAL     A            1 19.323 29.727 42.781       N
ATOM        2        CA     VAL     A            1 20.141 30.469 42.414       C
ATOM        3         C     VAL     A            1 21.664 29.857 42.548       C
ATOM        4         O     VAL     A            1 21.985 29.541 43.704       O
ATOM        5        CB     VAL     A            1 19.887 31.918 43.524       C
ATOM        6       CG1     VAL     A            1 20.656 32.850 42.999       C
ATOM        7       CG2     VAL     A            1 18.692 31.583 43.506       C
ATOM        8         N     LEU     A            2 22.284 30.338 41.520       N
ATOM        9        CA     LEU     A            2 23.773 29.985 41.873       C
ATOM       10         C     LEU     A            2 24.698 31.236 42.725       C


In [5]:
import requests
import pandas as pd
import numpy as np

# 1. Fetching 3D coordinates using PubChem's native SDF format
url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/caffeine/SDF?record_type=3d"

print("Fetching Caffeine 3D coordinates (SDF format) from PubChem...")
response = requests.get(url)

if response.status_code == 200:
    # Save the raw file locally
    with open("caffeine.sdf", "w", encoding="utf-8") as f:
        f.write(response.text)
    print("Success! 'caffeine.sdf' downloaded.\n")
    
    # 2. Parse the SDF coordinate block 
    # An SDF file stores spatial atom data starting right after the header lines
    atoms = []
    lines = response.text.split("\n")
    
    # Line index 3 tells us how many atoms are in the molecule
    header_counts = lines[3].split()
    num_atoms = int(header_counts[0])
    
    # The next lines contain the absolute X, Y, Z coordinates and the Element type
    for i in range(4, 4 + num_atoms):
        line = lines[i]
        parts = line.split()
        
        # If the file format spacing varies, parse positions safely
        x = float(line[0:10].strip())
        y = float(line[10:20].strip())
        z = float(line[20:30].strip())
        element = line[31:34].strip()
        
        atoms.append({
            "Atom_ID": i - 3,
            "Element": element,
            "X": x,
            "Y": y,
            "Z": z
        })

    df = pd.DataFrame(atoms)

    # 3. Analyze the Structure
    print("--- MOLECULAR ANALYSIS REPORT ---")
    print(f"Total Atoms in Caffeine ($C_8H_{{10}}N_4O_2$): {len(df)}")
    print("\nAtom Counts by Element:")
    print(df["Element"].value_counts())

    # Calculate the spatial dimensions (box size) to observe planarity
    max_x, min_x = df["X"].max(), df["X"].min()
    max_y, min_y = df["Y"].max(), df["Y"].min()
    max_z, min_z = df["Z"].max(), df["Z"].min()
    
    print(f"\n3D Bounding Box Dimensions:")
    print(f"X thickness: {max_x - min_x:.3f} Å")
    print(f"Y thickness: {max_y - min_y:.3f} Å")
    print(f"Z thickness: {max_z - min_z:.3f} Å")

else:
    print(f"Failed to fetch molecule. Status code: {response.status_code}")

Fetching Caffeine 3D coordinates (SDF format) from PubChem...
Success! 'caffeine.sdf' downloaded.

--- MOLECULAR ANALYSIS REPORT ---
Total Atoms in Caffeine ($C_8H_{10}N_4O_2$): 24

Atom Counts by Element:
Element
H    10
C     8
N     4
O     2
Name: count, dtype: int64

3D Bounding Box Dimensions:
X thickness: 7.326 Å
Y thickness: 6.362 Å
Z thickness: 1.800 Å


In [6]:
# Function to calculate distance between two atoms by their DataFrame index
def calculate_bond_length(df, atom_idx1, atom_idx2):
    atom1 = df.iloc[atom_idx1][["X", "Y", "Z"]].values
    atom2 = df.iloc[atom_idx2][["X", "Y", "Z"]].values
    
    distance = np.linalg.norm(atom1 - atom2)
    return distance

# Example: Check the distance between the first two atoms in your DataFrame
length = calculate_bond_length(df, 0, 1)
print(f"Bond length between Atom 0 and Atom 1: {length:.3f} Å")

Bond length between Atom 0 and Atom 1: 4.692 Å


In [7]:
df

,Atom_ID,Element,X,Y,Z
0,1,O,0.4700,2.5688,0.0006
1,2,O,-3.1271,-0.4436,-0.0003
2,3,N,-0.9686,-1.3125,0.0000
3,4,N,2.2182,0.1412,-0.0003
4,5,N,-1.3477,1.0797,-0.0001
5,6,N,1.4119,-1.9372,0.0002
6,7,C,0.8579,0.2592,-0.0008
7,8,C,0.3897,-1.0264,-0.0004
8,9,C,0.0307,1.4220,-0.0006
9,10,C,-1.9061,-0.2495,-0.0004


In [8]:
# Filter out Hydrogens to look at the main ring structure
heavy_atoms_df = df[df["Element"] != "H"]

print(f"Heavy atoms skeleton size: {len(heavy_atoms_df)} atoms")
print(heavy_atoms_df[["Element", "X", "Y", "Z"]].to_string(index=False))

Heavy atoms skeleton size: 14 atoms
Element       X       Y       Z
      O  0.4700  2.5688  0.0006
      O -3.1271 -0.4436 -0.0003
      N -0.9686 -1.3125  0.0000
      N  2.2182  0.1412 -0.0003
      N -1.3477  1.0797 -0.0001
      N  1.4119 -1.9372  0.0002
      C  0.8579  0.2592 -0.0008
      C  0.3897 -1.0264 -0.0004
      C  0.0307  1.4220 -0.0006
      C -1.9061 -0.2495 -0.0004
      C  2.5032 -1.1998  0.0003
      C -1.4276 -2.6960  0.0008
      C  3.1926  1.2061  0.0003
      C -2.2969  2.1881  0.0007
